# Импорт зависимостей

In [12]:
import sys, os
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.data.downloader import download_warc_files
from src.data.converter import convert_all_warc
from src.data.cleaner import clean_all_jsonl
from src.data.entropy import compute_dataset_entropy_dir, filter_dataset_dir
import src.tokenization.tokenizers as tk


# 3.1 Скачать датасет

## 1. Скачать часть Comon Crawl - WARC файлы

In [13]:
download_warc_files(config_path="../configs/config.yaml", raw_data_dir="../data/raw/common_crawl/warc")

19:55:36 | INFO     | Директория для сохранения: R:\Folders\1. master_degree_ITMO\semester 2\LABS\MNNA-2026\MNNA-2026-labs-DmitrievDM\data\raw\common_crawl\warc


Общий прогресс:   0%|          | 0/1 [00:00<?, ?it/s]

19:55:36 | INFO     | Файл CC-MAIN-20260605214811-20260606004811-00000.warc.gz уже существует. Пропускаем.


## 2. Конверитровать WARC файлы в текстовый формат.

In [14]:
convert_all_warc(input_dir="../data/raw/common_crawl/warc", output_dir="../data/raw/common_crawl/jsonl")

19:55:36 | INFO     | Найдено WARC-файлов: 1


Конвертация WARC:   0%|          | 0/1 [00:00<?, ?file/s]

19:55:36 | INFO     | Обработка файла: ..\data\raw\common_crawl\warc\CC-MAIN-20260605214811-20260606004811-00000.warc.gz
19:56:44 | INFO     | Файл ..\data\raw\common_crawl\warc\CC-MAIN-20260605214811-20260606004811-00000.warc.gz обработан: всего записей=63412, записано=20820, ошибок=0
19:56:44 | INFO     | Готово: ..\data\raw\common_crawl\jsonl\CC-MAIN-20260605214811-20260606004811-00000.jsonl
19:56:44 | INFO     | Обработка завершена
19:56:44 | INFO     | Файлов найдено: 1
19:56:44 | INFO     | Сконвертировано: 1
19:56:44 | INFO     | Пропущено: 0
19:56:44 | INFO     | Ошибок: 0


# 3.2 Очистка данных

In [15]:
clean_all_jsonl(input_dir="../data/raw/common_crawl/jsonl", output_dir="../data/processed/common_crawl/stage1_cleaned")

19:56:44 | INFO     | Обработка: ..\data\raw\common_crawl\jsonl\CC-MAIN-20260605214811-20260606004811-00000.jsonl
20:19:34 | INFO     | Файл ..\data\raw\common_crawl\jsonl\CC-MAIN-20260605214811-20260606004811-00000.jsonl обработан: прочитано=20820, оставлено объектов=10031, отброшено=10789, итоговых чанков записано=23996


***Примечания:*** *стоит отладить код, добавить прогресс-бар, посмотреть как работает фильтрация и нужна ли она, также выяснить причины очень долгого выполнения очистки данных*

# 3.3 Улучшить качество данных

## 1. GPT2 и энтропия

In [16]:
stats = compute_dataset_entropy_dir(
    input_dir="../data/processed/common_crawl/stage1_cleaned",
    output_dir="../data/metrics/common_crawl/entropy_stats",
    stats_path="../data/metrics/common_crawl/full_entropy_stats.json",
    pattern="*.jsonl",
    model_name="gpt2",
    text_field="text",
    batch_size=8,
    max_length=1024,
)

print(stats["info_density_nats_per_token"])

20:19:34 | INFO     | Using device: cuda
20:19:34 | INFO     | HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
20:19:34 | INFO     | HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/config.json "HTTP/1.1 200 OK"
20:19:34 | WARNING  | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
20:19:35 | INFO     | HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
20:19:35 | INFO     | HTTP Request: GET https://huggingface.co/api/models/gpt2/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
20:19:35 | INFO     | HTTP Request: GET https://huggingface.co/api/models/openai-community/gpt2/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
20:19:35 | INFO     | HTTP Request: GET https://huggingface.co/api/models/gpt2/tree/main?recursive=t

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

20:19:41 | INFO     | HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/generation_config.json "HTTP/1.1 200 OK"
20:19:43 | INFO     | Processing file: ..\data\processed\common_crawl\stage1_cleaned\CC-MAIN-20260605214811-20260606004811-00000.jsonl


CC-MAIN-20260605214811-20260606004811-00000.jsonl: 0batch [00:00, ?batch/s]

21:47:16 | INFO     | File CC-MAIN-20260605214811-20260606004811-00000.jsonl done. Objects: 23996, info density: 2.813749
21:47:16 | INFO     | Global info density: 2.813749


2.813748689229325


## 2. Удаление дубликатов и объектов с высокой или низкой энтропией

In [17]:
filter_dataset_dir(
    original_dir="../data/processed/common_crawl/stage1_cleaned",               # Папка с ИСХОДНЫМ датасетом
    metrics_dir="../data/metrics/common_crawl/entropy_stats",    # Папка, куда предыдущий код сохранил энтропию
    output_dir="../data/processed/common_crawl/stage2_filtered",       # Папка, куда сохранится ИТОГОВЫЙ чистый датасет
    lower_percentile=1.0,                  # Удалить 1% текстов с самой НИЗКОЙ энтропией
    upper_percentile=99.0                  # Удалить 1% текстов с самой ВЫСОКОЙ энтропией
)

21:47:16 | INFO     | Загрузка метрик из CC-MAIN-20260605214811-20260606004811-00000.jsonl...
21:47:16 | INFO     | Установлены границы энтропии: [0.5188, 6.2660]
21:47:18 | INFO     | Файл CC-MAIN-20260605214811-20260606004811-00000.jsonl обработан.
21:47:18 | INFO     | Статистика: {'total_valid_objects': 23996, 'kept': 22891, 'removed_duplicates': 641, 'removed_low_entropy': 237, 'removed_high_entropy': 227, 'skipped_empty_or_invalid': 0}
21:47:18 | INFO     | ========================================
21:47:18 | INFO     | ГЛОБАЛЬНАЯ СТАТИСТИКА ОЧИСТКИ:
21:47:18 | INFO     | total_valid_objects: 23996
21:47:18 | INFO     | kept: 22891
21:47:18 | INFO     | removed_duplicates: 641
21:47:18 | INFO     | removed_low_entropy: 237
21:47:18 | INFO     | removed_high_entropy: 227
21:47:18 | INFO     | skipped_empty_or_invalid: 0


{'total_valid_objects': 23996,
 'kept': 22891,
 'removed_duplicates': 641,
 'removed_low_entropy': 237,
 'removed_high_entropy': 227,
 'skipped_empty_or_invalid': 0}

# 3.4 Токенизация

In [18]:
from pathlib import Path
import random
import json

# =========================
# 1. Настройки
# =========================
# Папка, куда filter_dataset_dir сохранил очищенные .jsonl файлы
FILTERED_DATA_DIR = Path("../data/processed/common_crawl/stage2_filtered") 

TEXT_FIELD = "text"
SEED = 42
OUTPUT_DIR = Path("../data/tokenizers")

# Если данных очень много и не хватает оперативной памяти (RAM), 
# ограничьте количество текстов. Например, MAX_TEXTS = 200_000.
# Если хотите читать всё до конца, поставьте None.
MAX_TEXTS = 200_000 

# =========================
# 2. Простой цикл для чтения всех .jsonl файлов
# =========================
print(f"Поиск .jsonl файлов в {FILTERED_DATA_DIR}...")
jsonl_files = sorted(FILTERED_DATA_DIR.glob("*.jsonl"))

if not jsonl_files:
    raise FileNotFoundError(f"Не найдено .jsonl файлов в {FILTERED_DATA_DIR}")

print(f"Найдено файлов: {len(jsonl_files)}")
print("Чтение текстов...")

texts = []
for file_path in jsonl_files:
    with file_path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            
            try:
                obj = json.loads(line)
                text = obj.get(TEXT_FIELD, "")
                if text:
                    texts.append(str(text))
            except json.JSONDecodeError:
                continue
                
            # Прерываем чтение, если достигли лимита
            if MAX_TEXTS is not None and len(texts) >= MAX_TEXTS:
                break
                
    if MAX_TEXTS is not None and len(texts) >= MAX_TEXTS:
        print(f"Достигнут лимит в {MAX_TEXTS} текстов. Останавливаем чтение.")
        break

print(f"Итого загружено текстов: {len(texts)}")
assert len(texts) > 0, "Список текстов пуст!"

# Выбираем случайный объект для демонстрации
rng = random.Random(SEED)
sample_idx = rng.randrange(len(texts))
sample_text = texts[sample_idx]

print("-" * 60)
print(f"Индекс случайного объекта: {sample_idx}")
print(f"Пример текста (первые 150 символов): {sample_text[:150]}")

Поиск .jsonl файлов в ..\data\processed\common_crawl\stage2_filtered...
Найдено файлов: 1
Чтение текстов...
Итого загружено текстов: 22891
------------------------------------------------------------
Индекс случайного объекта: 20952
Пример текста (первые 150 символов): Why Choose ZOHO CRM Customer Stories & Testimonials ZOHO CRM Features Learn More ZOHO CRM Website ZOHO Begin Website 03 - Service Delivery Sales ZOHO 


## 3.4.1 Символьная токенизация

In [19]:
print("-" * 60)
print("Обучение символьного токенизатора...")
char_token2id = tk.fit_char_tokenizer(texts, max_samples=None, seed=SEED)
char_ids = tk.encode_char(sample_text, char_token2id, add_special=True)

print("1. Токенизация по символам")
print(f"Размер словаря: {len(char_token2id)}")
print(f"Размер последовательности случайного объекта: {len(char_ids)}")
# Сохранение
char_path = tk.save_vocab(char_token2id, OUTPUT_DIR / "char_tokenizer.json")

------------------------------------------------------------
Обучение символьного токенизатора...
1. Токенизация по символам
Размер словаря: 166
Размер последовательности случайного объекта: 2854


## 3.4.2 Токенизация по словам

In [20]:
print("-" * 60)
print("Обучение словесного токенизатора...")
word_token2id = tk.fit_word_tokenizer(
    texts,
    max_samples=100_000, 
    max_vocab_size=30_000,
    min_freq=2,
    seed=SEED
)
word_ids = tk.encode_word(sample_text, word_token2id, add_special=True)

print("2. Токенизация по словам")
print(f"Размер словаря: {len(word_token2id)}")
print(f"Размер последовательности случайного объекта: {len(word_ids)}")
# Сохранение
word_path = tk.save_vocab(word_token2id, OUTPUT_DIR / "word_tokenizer.json")

------------------------------------------------------------
Обучение словесного токенизатора...
2. Токенизация по словам
Размер словаря: 30004
Размер последовательности случайного объекта: 396


## 3.4.3 BPE

In [21]:
print("-" * 60)
print("Обучение BPE токенизатора (может занять пару минут)...")
bpe_tokenizer = tk.train_bpe_tokenizer(
    texts,
    max_samples=100_000,
    vocab_size=10_000,
    min_freq=2,
    seed=SEED
)
bpe_ids = tk.encode_bpe(sample_text, bpe_tokenizer, add_special=True)

print("3. BPE-токенизация")
print(f"Размер словаря: {bpe_tokenizer.get_vocab_size()}")
print(f"Размер последовательности случайного объекта: {len(bpe_ids)}")
# Сохранение
bpe_path = tk.save_bpe_tokenizer(bpe_tokenizer, OUTPUT_DIR / "bpe_tokenizer.json")

------------------------------------------------------------
Обучение BPE токенизатора (может занять пару минут)...
3. BPE-токенизация
Размер словаря: 10000
Размер последовательности случайного объекта: 575


# 3.5 Обучение на wikitext

## 3.5.1 Скачивание датасета Wikitext

In [22]:
from datasets import load_dataset
import json
from pathlib import Path

raw_wiki_dir = Path("../data/raw/wikitext")
raw_wiki_dir.mkdir(parents=True, exist_ok=True)

# Использовать можно либо: wikitext-103-raw-v1 либо если долго: wikitext-2-raw-v1.
print("Скачивание wikitext...")
wiki = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")

for split in ["train", "validation", "test"]:
    out_path = raw_wiki_dir / f"{split}.jsonl"
    print(f"Сохранение {split} в {out_path}...")
    with out_path.open("w", encoding="utf-8") as f:
        for item in wiki[split]:
            text = item["text"].strip()
            if text:
                f.write(json.dumps({"text": text}, ensure_ascii=False) + "\n")
                
print("Скачивание и конвертация в JSONL завершены.")

Скачивание wikitext...


21:47:40 | INFO     | HTTP Request: HEAD https://huggingface.co/datasets/Salesforce/wikitext/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
21:47:41 | INFO     | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/Salesforce/wikitext/b08601e04326c79dfdd32d625aee71d232d685c3/README.md "HTTP/1.1 200 OK"
21:47:41 | INFO     | HTTP Request: HEAD https://huggingface.co/datasets/Salesforce/wikitext/resolve/b08601e04326c79dfdd32d625aee71d232d685c3/wikitext.py "HTTP/1.1 404 Not Found"
21:47:41 | INFO     | HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/Salesforce/wikitext/Salesforce/wikitext.py "HTTP/1.1 404 Not Found"
21:47:42 | INFO     | HTTP Request: GET https://huggingface.co/api/datasets/Salesforce/wikitext/revision/b08601e04326c79dfdd32d625aee71d232d685c3 "HTTP/1.1 200 OK"
21:47:42 | INFO     | HTTP Request: HEAD https://huggingface.co/datasets/Salesforce/wikitext/resolve/b08601e04326c79dfdd32d625aee71d232d685c3/.huggingf

Сохранение train в ..\data\raw\wikitext\train.jsonl...
Сохранение validation в ..\data\raw\wikitext\validation.jsonl...
Сохранение test в ..\data\raw\wikitext\test.jsonl...
Скачивание и конвертация в JSONL завершены.


## 3.5.2 Очистка, Энтропия и Фильтрация

In [23]:
print("1. Очистка данных wikitext...")
# clean_all_jsonl(input_dir="../data/raw_wikitext", output_dir="../data/cleaned_wikitext")
import shutil
from pathlib import Path

raw_dir = Path("../data/raw/wikitext")
cleaned_dir = Path("../data/processed/wikitext/stage1_cleaned")

cleaned_dir.mkdir(parents=True, exist_ok=True)

print("Копирование исходных файлов wikitext в обход клинера...")
for f in raw_dir.glob("*.jsonl"):
    shutil.copy(f, cleaned_dir / f.name)
    
# Проверка
print("\nПроверка количества строк в cleaned_wikitext:")
for f in sorted(cleaned_dir.glob("*.jsonl")):
    lines = sum(1 for _ in f.open("r", encoding="utf-8"))
    print(f"  {f.name}: {lines} строк")
# ===========================================================
print("\n2. Вычисление энтропии (GPT-2)...")
stats = compute_dataset_entropy_dir(
    input_dir="../data/processed/wikitext/stage1_cleaned",
    output_dir="../data/metrics/wikitext/entropy_stats",
    stats_path="../data/metrics/wikitext/full_entropy_stats.json",
    pattern="*.jsonl",
    model_name="gpt2",
    text_field="text",
    batch_size=24,
    max_length=1024,
)
print(f"Информационная плотность wikitext: {stats['info_density_nats_per_token']:.4f}")

print("\n3. Фильтрация дубликатов и экстремальной энтропии...")
filter_dataset_dir(
    original_dir="../data/processed/wikitext/stage1_cleaned",
    metrics_dir="../data/metrics/wikitext/entropy_stats",
    output_dir="../data/processed/wikitext/stage2_filtered",
    lower_percentile=1.0,
    upper_percentile=99.0
)
print("\nПайплайн подготовки wikitext завершен!")

21:47:44 | INFO     | Using device: cuda


1. Очистка данных wikitext...
Копирование исходных файлов wikitext в обход клинера...

Проверка количества строк в cleaned_wikitext:
  test.jsonl: 2891 строк
  train.jsonl: 23767 строк
  validation.jsonl: 2461 строк

2. Вычисление энтропии (GPT-2)...


21:47:44 | INFO     | HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/config.json "HTTP/1.1 200 OK"
21:47:45 | INFO     | HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
21:47:45 | INFO     | HTTP Request: GET https://huggingface.co/api/models/gpt2/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
21:47:45 | INFO     | HTTP Request: GET https://huggingface.co/api/models/openai-community/gpt2/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
21:47:45 | INFO     | HTTP Request: GET https://huggingface.co/api/models/gpt2/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
21:47:45 | INFO     | HTTP Request: GET https://huggingface.co/api/models/openai-community/gpt2/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
21:47:46 | INFO     | HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/config.json "HTTP

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

21:47:46 | INFO     | HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/generation_config.json "HTTP/1.1 200 OK"
21:47:48 | INFO     | Processing file: ..\data\processed\wikitext\stage1_cleaned\test.jsonl


...

validation.jsonl: 0batch [00:00, ?batch/s]

22:17:49 | INFO     | File validation.jsonl done. Objects: 2461, info density: 3.894777
22:17:49 | INFO     | Global info density: 3.882923
22:17:49 | INFO     | Загрузка метрик из test.jsonl...
22:17:49 | INFO     | Установлены границы энтропии: [2.9024, 10.0964]


Информационная плотность wikitext: 3.8829

3. Фильтрация дубликатов и экстремальной энтропии...


22:17:49 | INFO     | Файл test.jsonl обработан.
22:17:49 | INFO     | Статистика: {'total_valid_objects': 2891, 'kept': 2739, 'removed_duplicates': 99, 'removed_low_entropy': 28, 'removed_high_entropy': 25, 'skipped_empty_or_invalid': 0}
22:17:49 | INFO     | Загрузка метрик из train.jsonl...
22:17:49 | INFO     | Установлены границы энтропии: [2.9167, 9.1384]
22:17:49 | INFO     | Файл train.jsonl обработан.
22:17:49 | INFO     | Статистика: {'total_valid_objects': 23767, 'kept': 21319, 'removed_duplicates': 2054, 'removed_low_entropy': 228, 'removed_high_entropy': 166, 'skipped_empty_or_invalid': 0}
22:17:49 | INFO     | Загрузка метрик из validation.jsonl...
22:17:50 | INFO     | Установлены границы энтропии: [2.9621, 9.2725]
22:17:50 | INFO     | Файл validation.jsonl обработан.
22:17:50 | INFO     | Статистика: {'total_valid_objects': 2461, 'kept': 2317, 'removed_duplicates': 100, 'removed_low_entropy': 25, 'removed_high_entropy': 19, 'skipped_empty_or_invalid': 0}
22:17:50 | INF


Пайплайн подготовки wikitext завершен!


## 3.5.3 Реализация Packed Batching и LightningDataModule

In [24]:
import torch
import pytorch_lightning as pl
from torch.utils.data import DataLoader, Dataset

def create_packed_batches(texts, tokenizer, max_length=512):
    """
    Packed batching с локальной маской:
    0 — PAD, 1 — первый объект, 2 — второй объект и т.д.
    """
    packs = []

    current_input_ids = []
    current_mask = []
    segment_id = 1  # локальный ID внутри текущего пака

    pad_id = tokenizer.token_to_id("<pad>")
    eos_id = tokenizer.token_to_id("<eos>")

    for text in texts:
        ids = tokenizer.encode(text).ids

        if eos_id is not None:
            ids.append(eos_id)

        current_input_ids.extend(ids)
        current_mask.extend([segment_id] * len(ids))
        segment_id += 1

        # Набиваем полные паки
        while len(current_input_ids) >= max_length:
            pack_input_ids = current_input_ids[:max_length]
            pack_mask = current_mask[:max_length]

            # Нормализуем маску: 1, 2, 3...
            # (на случай, если в пак попали куски одного и того же документа)
            unique_ids = sorted(set(pack_mask))
            id_map = {old: new for new, old in enumerate(unique_ids, start=1)}
            pack_mask = [id_map[m] for m in pack_mask]

            packs.append({
                "input_ids": pack_input_ids,
                "mask": pack_mask
            })

            # Остаток переносим в новый пак
            remaining_ids = current_input_ids[max_length:]
            remaining_mask = current_mask[max_length:]

            # Сбрасываем segment_id для нового пака
            # Остаток принадлежит тому же документу, что и конец предыдущего пака
            current_input_ids = remaining_ids
            current_mask = remaining_mask
            segment_id = max(id_map.values()) + 1 if remaining_mask else 1

    # Последний неполный пак — паддим до max_length
    if len(current_input_ids) > 0:
        pad_len = max_length - len(current_input_ids)

        # Нормализуем маску перед паддингом
        unique_ids = sorted(set(current_mask))
        id_map = {old: new for new, old in enumerate(unique_ids, start=1)}
        current_mask = [id_map[m] for m in current_mask]

        pack_input_ids = current_input_ids + [pad_id] * pad_len
        pack_mask = current_mask + [0] * pad_len  # 0 для PAD

        packs.append({
            "input_ids": pack_input_ids,
            "mask": pack_mask
        })

    return packs


class PackedWikiTextDataset(Dataset):
    def __init__(self, packs):
        self.packs = packs
        
    def __len__(self):
        return len(self.packs)
        
    def __getitem__(self, idx):
        pack = self.packs[idx]
        return {
            "input_ids": torch.tensor(pack["input_ids"], dtype=torch.long),
            "mask": torch.tensor(pack["mask"], dtype=torch.long),
        }


class WikiTextDataModule(pl.LightningDataModule):
    def __init__(self, data_dir, tokenizer_path, max_length=512, batch_size=4):
        super().__init__()
        self.data_dir = Path(data_dir)
        self.tokenizer_path = tokenizer_path
        self.max_length = max_length
        self.batch_size = batch_size
        
    def setup(self, stage=None):
        # Загружаем BPE, обученный на Common Crawl!
        self.tokenizer = tk.load_bpe_tokenizer(self.tokenizer_path)
        
        self.train_packs = self._pack_split("train.jsonl")
        self.val_packs = self._pack_split("validation.jsonl")
        
    def _pack_split(self, filename):
        texts = []
        file_path = self.data_dir / filename
        if not file_path.exists():
            return []
            
        with file_path.open("r", encoding="utf-8") as f:
            for line in f:
                obj = json.loads(line)
                if obj.get("text"):
                    texts.append(obj["text"])
                    
        return create_packed_batches(texts, self.tokenizer, self.max_length)
        
    def train_dataloader(self):
        dataset = PackedWikiTextDataset(self.train_packs)
        return DataLoader(dataset, batch_size=self.batch_size, shuffle=True)
        
    def val_dataloader(self):
        dataset = PackedWikiTextDataset(self.val_packs)
        return DataLoader(dataset, batch_size=self.batch_size)

W0818 22:17:55.988000 33200 .venv\Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


## 3.5.4 Демонстрация

In [25]:
# Инициализируем DataModule
import json
from pathlib import Path

dm = WikiTextDataModule(
    data_dir="../data/processed/wikitext/stage2_filtered",
    tokenizer_path="../data/tokenizers/bpe_tokenizer.json", # BPE от Common Crawl
    max_length=512,
    batch_size=2
)

dm.setup()

# Берем один батч из train_dataloader
train_loader = dm.train_dataloader()
batch = next(iter(train_loader))

print("Размер батча (input_ids):", batch["input_ids"].shape)
print("Размер батча (mask):", batch["mask"].shape)

# Посмотрим на первый объект в батче
print("\n--- Первый объект в батче ---")
print("Первые 20 input_ids:", batch["input_ids"][0][:20].tolist(), "...")
print("Первые 20 mask:", batch["mask"][0][:20].tolist(), "...")

# Проверим, как работает маска в конце последовательности (где должны быть PAD токены)
print("\nМаска в конце первого объекта (должны быть нули для PAD):")
print("mask:", batch["mask"][0][-20:].tolist())

# Проверим, есть ли склейка разных объектов (значения маски 1, 2, 3...)
unique_segments = torch.unique(batch["mask"][0]).tolist()
print(f"\nУникальные ID объектов в первом батче (0 - это PAD): {unique_segments}")
print("Если в списке есть числа больше 1 (например, 1, 2, 3), значит packed batching успешно склеил несколько текстов в один!")

Размер батча (input_ids): torch.Size([2, 512])
Размер батча (mask): torch.Size([2, 512])

--- Первый объект в батче ---
Первые 20 input_ids: [446, 2314, 419, 295, 17, 3, 32, 32, 305, 2069, 9611, 32, 32, 3, 290, 260, 2069, 591, 7008, 167] ...
Первые 20 mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2] ...

Маска в конце первого объекта (должны быть нули для PAD):
mask: [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]

Уникальные ID объектов в первом батче (0 - это PAD): [1, 2, 3, 4]
Если в списке есть числа больше 1 (например, 1, 2, 3), значит packed batching успешно склеил несколько текстов в один!


In [26]:
# Берём самый последний пак из train
last_pack = dm.train_packs[-1]

print("Маска в конце последнего пака (должны быть нули - PAD):")
print(last_pack["mask"][-30:])

unique_segments = sorted(list(set(last_pack["mask"])))
print(f"\nУникальные ID в последнем паке (0 - это PAD): {unique_segments}")

Маска в конце последнего пака (должны быть нули - PAD):
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

Уникальные ID в последнем паке (0 - это PAD): [0, 1, 2]
